In [ ]:
# Cell 1: Install required packages for Shiny and Computer Vision
!pip install shiny shinylive shinywidgets ultralytics opencv-python pillow matplotlib

In [1]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image

# Import Shiny components
from shiny import App, render, ui, reactive
from shinywidgets import output_widget, render_widget
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. LOAD YOUR TRAINED MODEL
# Make sure your 'best.pt' file is placed in your repository folder
MODEL_PATH = Path(__file__).parent / "best.pt" if "__file__" in locals() else "best.pt"

try:
    model = YOLO(MODEL_PATH)
except Exception as e:
    print(f"Warning: Could not load model from {MODEL_PATH}. App will run in simulation mode. Error: {e}")
    model = None

# 2. USER INTERFACE (UI) DESIGN
app_ui = ui.page_sidebar(
    ui.sidebar(
        ui.h3("Agronomic Inputs"),
        ui.input_file("uploaded_image", "1. Upload Corn Ear Image", accept=[".jpg", ".jpeg", ".png"]),
        ui.input_numeric("test_weight", "2. Test Weight (lbs/bu)", value=56.0, min=40.0, max=70.0, step=0.1),
        ui.input_numeric("row_spacing", "3. Row Spacing (inches)", value=30.0, min=10.0, max=40.0, step=0.5),
        ui.input_numeric("seed_spacing", "4. Seed Spacing (inches)", value=6.0, min=2.0, max=15.0, step=0.1),
        ui.input_numeric("total_hectares", "5. Total Field Area (Hectares)", value=10.0, min=0.1, step=0.5),
        ui.input_action_button("calculate_btn", "Calculate Yield", class_="btn-primary w-100 mt-3"),
        title="Kernel Counter Settings"
    ),
    ui.layout_columns(
        ui.value_box(
            "Detected Kernels per Ear",
            ui.output_text("txt_kernel_count"),
            theme="bg-gradient-blue-purple"
        ),
        ui.value_box(
            "Estimated Plant Population",
            ui.output_text("txt_population"),
            theme="bg-gradient-green-blue"
        ),
        ui.value_box(
            "Estimated Field Yield",
            ui.output_text("txt_total_yield"),
            theme="bg-gradient-orange-red"
        ),
        col_widths=[4, 4, 4]
    ),
    ui.card(
        ui.card_header("AI Visual Verification (Detected Kernels)"),
        ui.output_plot("plot_prediction"),
        full_screen=True
    ),
    title="Precision Ag: AI Corn Kernel Counter & Yield Estimator"
)

# 3. SERVER LOGIC
def server(input, output, session):
    
    # Reactive event that triggers only when the farmer clicks "Calculate Yield"
    @reactive.calc
    @reactive.event(input.calculate_btn)
    def run_analysis():
        # Ensure an image has been uploaded
        image_file = input.uploaded_image()
        if not image_file:
            return None
            
        # Read the uploaded image using OpenCV
        img_path = image_file[0]["datapath"]
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Run AI Inference
        if model is not None:
            results = model(img_rgb)
            kernel_count = len(results[0].boxes)
            # Render the bounding boxes onto the image
            annotated_img = results[0].plot(labels=False, conf=False) 
        else:
            # Fallback mock data if model isn't found locally yet
            kernel_count = 450
            annotated_img = img_rgb.copy()
            cv2.putText(annotated_img, "Simulation Mode", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            
        # AGRONOMIC CALCULATIONS
        # 1. Calculate Plant Population per Hectare
        # Space per single plant in square inches
        sq_inches_per_plant = input.row_spacing() * input.seed_spacing()
        # 1 Hectare = 15,500,031 square inches
        plants_per_hectare = 15500031.0 / sq_inches_per_plant
        
        # 2. Calculate Yield Component Method (Bushels per Hectare)
        # Standard formulation assuming 90,000 kernels per bushel at standard test weight
        # Adjusting dynamically for user input test weight variations
        kernels_per_bushel = (90000.0 * (56.0 / input.test_weight()))
        bushels_per_hectare = (plants_per_hectare * kernel_count) / kernels_per_bushel
        
        # 3. Total production for specified hectares
        total_production_bushels = bushels_per_hectare * input.total_hectares()
        
        return {
            "annotated_image": annotated_img,
            "kernel_count": kernel_count,
            "population": int(plants_per_hectare),
            "total_yield": round(total_production_bushels, 1)
        }

    # Output text: Kernel Count
    @render.text
    def txt_kernel_count():
        res = run_analysis()
        return f"{res['kernel_count']} kernels" if res else "Upload image and click calculate"

    # Output text: Plant Population
    @render.text
    def txt_population():
        res = run_analysis()
        return f"{res['population']:,} plants/ha" if res else "Waiting for inputs..."

    # Output text: Final calculated production yield
    @render.text
    def txt_total_yield():
        res = run_analysis()
        return f"{res['total_yield']:,} Bushels total" if res else "Waiting for inputs..."

    # Output plot: Image displaying bounding boxes
    @render.plot
    def plot_prediction():
        res = run_analysis()
        fig, ax = plt.subplots(figsize=(10, 8))
        if res:
            ax.imshow(res["annotated_image"])
        else:
            # Placeholder before upload
            ax.text(0.5, 0.5, "Please upload a clear ear image\nand click 'Calculate Yield'", 
                    ha='center', va='center', fontsize=14, color='gray')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        ax.axis("off")
        plt.tight_layout()
        return fig

# 4. RUN THE APP ENVIRONMENT
app = App(app_ui, server)

In [2]:
!pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [3]:
# Cell 3: View the app dynamically inside your Jupyter Notebook
from shiny import run_app
run_app(app, port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop